# Residual RK2 Prefix Control Evaluation

## TL;DR

Final analysis entry point for the 12-case, `N=0..15` source-referenced residual RK2 sweep (192 unique outputs). Until images, LPIPS, and human scores are complete, this notebook reports **pending** rather than selecting a duration.

## Context and Methods

The edited state is a residual from the aligned source-inversion trajectory. At every controlled RK2 step:

\[
d_{mid}=d_i+M(	frac12 h v_1-(s_{mid}-s_i)),\quad x_{mid}=s_{mid}+d_{mid}
\]
\[
d_{i+1}=d_i+M(hv_2-(s_{i+1}-s_i)),\quad x_{i+1}=s_{i+1}+d_{i+1}.
\]

`N` is the **projection duration**: `N=0` is uncontrolled; `N>0` controls steps `0..N-1`. The primary sweep uses the oracle GT mask and no image-KV injection.

### Key assumptions

- Source endpoints and midpoints are aligned to denoising indices.
- Inside-mask pixel metrics measure activity, not semantic success.
- Human local-edit and preservation scores are separate.
- Any selected `N` is global; case-specific tuning is prohibited.

In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
pd.set_option("display.max_columns", 80)
ROOT=Path.cwd().resolve()
while not (ROOT/"pyproject.toml").exists():
    if ROOT==ROOT.parent: raise RuntimeError("repository root not found")
    ROOT=ROOT.parent
RUN_ROOT=ROOT/"core/results/control_operations/residual_rk2_prefix_sweep"
EVAL_ROOT=ROOT/"core/results/control_operations_eval/residual_rk2_prefix_sweep"
METRICS_PATH=EVAL_ROOT/"unified_image_metrics.csv"
REVIEW_PATH=EVAL_ROOT/"manual_review_scores.csv"
EXPECTED_CASES=12; DURATIONS=list(range(16)); EXPECTED=192
print("repo:", ROOT)

repo: /Users/pt623/Documents/career-vault-resume/applications/hkust-harry-yang/part-level-overediting


## Data Integrity

A final analysis requires all 192 images, all residual metric/LPIPS rows, and all 192 valid human-score rows.

In [2]:
manifest=pd.read_json(ROOT/"core/data/partedit_subset/pilot_12_manifest.json")
expected_images=[RUN_ROOT/f"duration_{n:02d}"/uid/"seed_000/img_0.jpg" for n in DURATIONS for uid in manifest.case_uid]
missing_images=[x for x in expected_images if not x.is_file()]
metrics=pd.read_csv(METRICS_PATH) if METRICS_PATH.is_file() else pd.DataFrame()
review=pd.read_csv(REVIEW_PATH,dtype=str,keep_default_na=False) if REVIEW_PATH.is_file() else pd.DataFrame()
residual=metrics[metrics.method=="residual_rk2"].copy() if not metrics.empty and "method" in metrics else pd.DataFrame()
lpips_count=int(pd.to_numeric(residual.get("outside_mask_lpips"),errors="coerce").notna().sum()) if not residual.empty else 0
valid_review=0
if not review.empty and {"local_edit_success_0_2","non_target_preservation_0_2"}.issubset(review):
    a=pd.to_numeric(review.local_edit_success_0_2,errors="coerce"); b=pd.to_numeric(review.non_target_preservation_0_2,errors="coerce")
    valid_review=int((a.isin([0,1,2])&b.isin([0,1,2])).sum())
audit=pd.DataFrame([
{"check":"Residual images","found":EXPECTED-len(missing_images),"expected":EXPECTED},
{"check":"Residual metric rows","found":len(residual),"expected":EXPECTED},
{"check":"Residual LPIPS values","found":lpips_count,"expected":EXPECTED},
{"check":"Completed human rows","found":valid_review,"expected":EXPECTED}])
audit["complete"]=audit.found==audit.expected
display(audit); DATA_READY=bool(audit.complete.all())
display(Markdown("**Analysis status:** "+("complete" if DATA_READY else "pending; see missing evidence above.")))

,check,found,expected,complete
0,Residual images,0,192,False
1,Residual metric rows,0,192,False
2,Residual LPIPS values,0,192,False
3,Completed human rows,0,192,False


**Analysis status:** pending; see missing evidence above.

## Quantitative Results

Non-target preservation uses outside-mask L1 (lower), PSNR (higher), global-SSIM proxy (higher), and LPIPS (lower). Inside-mask L1/PSNR/SSIM are descriptive activity measures, not prompt-adherence scores.

In [3]:
contract=pd.DataFrame([
("outside_mask_l1_aux","non-target preservation","lower"),
("outside_mask_psnr","non-target preservation","higher"),
("outside_mask_global_ssim","non-target preservation","higher"),
("outside_mask_lpips","non-target preservation","lower"),
("inside_mask_l1_aux","target activity","descriptive"),
("inside_mask_psnr","target activity","descriptive"),
("inside_mask_global_ssim","target activity","descriptive")],columns=["metric","role","direction"])
display(contract)
summary=pd.DataFrame()
if residual.empty: display(Markdown("Automatic curves are **pending**."))
else:
    cols=contract.metric.tolist()
    for col in cols: residual[col]=pd.to_numeric(residual[col],errors="coerce")
    summary=residual.groupby("duration",as_index=False)[cols].mean(); display(summary.round(4))
    fig,axs=plt.subplots(2,2,figsize=(13,8),constrained_layout=True)
    specs=[("outside_mask_l1_aux","Outside L1"),("outside_mask_psnr","Outside PSNR"),("outside_mask_global_ssim","Outside SSIM proxy"),("outside_mask_lpips","Outside LPIPS")]
    baseline=metrics[metrics.method=="original_fys"] if "method" in metrics else pd.DataFrame()
    for ax,(col,title) in zip(axs.ravel(),specs):
        ax.plot(summary.duration,summary[col],marker="o",color="#16697a",label="Residual RK2")
        if not baseline.empty:
            ax.axhline(pd.to_numeric(baseline[col],errors="coerce").mean(),ls="--",color="#b54834",label="Original FYS-TDM")
        ax.set(title=title,xlabel="Projection duration N"); ax.grid(alpha=.25); ax.legend()
    plt.show()

,metric,role,direction
0,outside_mask_l1_aux,non-target preservation,lower
1,outside_mask_psnr,non-target preservation,higher
2,outside_mask_global_ssim,non-target preservation,higher
3,outside_mask_lpips,non-target preservation,lower
4,inside_mask_l1_aux,target activity,descriptive
5,inside_mask_psnr,target activity,descriptive
6,inside_mask_global_ssim,target activity,descriptive


Automatic curves are **pending**.

## Human Evaluation

Every output receives independent 0-2 scores for local-edit success and non-target preservation. Preservation alone is not semantic success.

In [4]:
human=pd.DataFrame()
if valid_review!=EXPECTED: display(Markdown("Human analysis is **pending** until all 192 rows are scored."))
else:
    review["duration"]=pd.to_numeric(review.duration).astype(int)
    for c in ["local_edit_success_0_2","non_target_preservation_0_2"]: review[c]=pd.to_numeric(review[c]).astype(int)
    review["joint"]=(review.local_edit_success_0_2>=1)&(review.non_target_preservation_0_2>=1)
    human=review.groupby("duration",as_index=False).agg(local_edit=("local_edit_success_0_2","mean"),preservation=("non_target_preservation_0_2","mean"),joint_success=("joint","mean")); human.joint_success*=100
    display(human.round(3))
    fig,axs=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
    axs[0].plot(human.duration,human.local_edit,marker="o",label="Local edit"); axs[0].plot(human.duration,human.preservation,marker="o",label="Preservation"); axs[0].set(xlabel="Projection duration N",ylabel="Mean score (0-2)",ylim=(0,2.05)); axs[0].legend(); axs[0].grid(alpha=.25)
    axs[1].plot(human.duration,human.joint_success,marker="o",color="#3f7d58"); axs[1].set(xlabel="Projection duration N",ylabel="Cases (%)",ylim=(0,100),title="Both scores >= 1"); axs[1].grid(alpha=.25)
    plt.show()

Human analysis is **pending** until all 192 rows are scored.

## Qualitative Results

One row per case at frozen durations. This exposes semantic failures that pixel preservation cannot detect.

In [5]:
selected=[0,1,2,4,8,15]
def thumb(path,size=(160,160)): return Image.open(path).convert("RGB").resize(size,Image.Resampling.LANCZOS)
if missing_images: display(Markdown("Qualitative grid is **pending** because residual images are incomplete."))
else:
    tw,th=160,160; labelw=230; hh=30; rows=[]
    for r in manifest.to_dict("records"):
        ims=[thumb(ROOT/r["source_image"])]+[thumb(RUN_ROOT/f"duration_{n:02d}"/r["case_uid"]/"seed_000/img_0.jpg") for n in selected]
        canvas=Image.new("RGB",(labelw+tw*len(ims),th+hh),"white"); d=ImageDraw.Draw(canvas); d.text((5,6),f"{r['case_uid']} | {r['part']} -> {r['edit']} | {r['part_size']}",fill="black")
        for j,(im,label) in enumerate(zip(ims,["source",*[f"N={n}" for n in selected]])):
            x=labelw+j*tw; d.text((x+4,6),label,fill="black"); canvas.paste(im,(x,hh))
        rows.append(canvas)
    sheet=Image.new("RGB",(rows[0].width,sum(x.height for x in rows)),"white"); y=0
    for row in rows: sheet.paste(row,(0,y)); y+=row.height
    display(sheet)

Qualitative grid is **pending** because residual images are incomplete.

## Takeaways

The final conclusion must jointly test whether residual RK2 improves preservation over Original FYS-TDM, where semantic edit success collapses, whether one duration works across part sizes, and whether midpoint-consistent control improves endpoint-only projection. These conclusions remain **pending** until the integrity audit is complete.

In [6]:
if DATA_READY:
    best_lpips=int(summary.loc[summary.outside_mask_lpips.idxmin(),"duration"]); best_joint=int(human.loc[human.joint_success.idxmax(),"duration"])
    display(Markdown(f"Diagnostic only: lowest outside LPIPS at **N={best_lpips}**; highest joint human success at **N={best_joint}**. Check part-size strata and failures before selection."))
else: display(Markdown("**No empirical conclusion is emitted while evidence is incomplete.**"))

**No empirical conclusion is emitted while evidence is incomplete.**

## Reproduction

```bash
python core/scripts/run_residual_rk2_prefix_sweep.py --manifest core/data/partedit_subset/pilot_12_manifest.json --all-cases --durations 0-15 --seed 0 --execute
python core/scripts/evaluate_residual_rk2_prefix_sweep.py --lpips require
python core/scripts/build_residual_rk2_manual_review.py
python core/scripts/build_residual_rk2_manual_review.py --validate core/results/control_operations_eval/residual_rk2_prefix_sweep/manual_review_scores.csv
python -m jupyter nbconvert --execute --to notebook --inplace core/notebooks/10_evaluate_residual_rk2_prefix_sweep.ipynb
```

Full setup is documented in the repository README.